In [0]:
%run ../config/config

In [0]:
from pathlib import Path
import yaml

def quote_identifier(identifier: str) -> str:
    return f"`{identifier.replace('`', '``')}`"

catalog = quote_identifier(CATALOG)
schemas = [STAGING_SCHEMA, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]

for schema_name in schemas:
    schema = quote_identifier(schema_name)

    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}
    """)

print("Schemas created.")

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS
    {catalog}.{STAGING_SCHEMA}.{STAGING_VOLUME}
""")

print("Staging volume created")

gold = f"{catalog}.{quote_identifier(schemas[2])}"

security_group_sql = SECURITY_GROUP.replace("'", "''")

spark.sql(f"""
    CREATE OR REPLACE FUNCTION {gold}.filter_products(
        countries STRING
    )
    RETURNS BOOLEAN
    RETURN
        is_account_group_member('{security_group_sql}')
        AND countries LIKE '%Ukraine%'
""")

spark.sql(f"""
    CREATE OR REPLACE FUNCTION {gold}.mask_entry_creator(
        creator STRING
    )
    RETURNS STRING
    RETURN
        CASE
            WHEN is_account_group_member('{security_group_sql}')
                THEN creator
            ELSE 'REDACTED'
        END
""")

print("Security functions created.")

principal_sql = quote_identifier(PRINCIPAL)

spark.sql(f"""
    GRANT USE CATALOG
    ON CATALOG {catalog}
    TO {principal_sql}
""")

if GOLD_READ_ACCESS:
    spark.sql(f"""
        GRANT USE SCHEMA
        ON SCHEMA {gold}
        TO {principal_sql}
    """)

    spark.sql(f"""
        GRANT SELECT
        ON SCHEMA {gold}
        TO {principal_sql}
    """)

for function_name in ("filter_products", "mask_entry_creator"):
    spark.sql(f"""
        GRANT EXECUTE
        ON FUNCTION {gold}.{quote_identifier(function_name)}
        TO {principal_sql}
    """)

print("Permissions configured.")